# Clean the data and build customer features

Apply the retailer's mapping file to convert the raw data into the standard format, validate it, and build one row per customer.

## 1. Imports and load the Parquet file



In [1]:
import json
import pandas as pd

raw = pd.read_parquet("../data/raw/online_retail_II.parquet")
print("Raw data:", raw.shape)

Raw data: (1067371, 8)


## 2. Load the mapping file



In [2]:
with open("../mappings/online_retail_ii.json") as f:
    mapping = json.load(f)

print("Source:", mapping["source_name"])
print("Column mapping:", mapping["column_mapping"])

Source: Online Retail II (UCI) - UK online retailer
Column mapping: {'Invoice': 'order_id', 'StockCode': 'product_id', 'Description': 'product_name', 'Quantity': 'quantity', 'InvoiceDate': 'order_date', 'Price': 'unit_price', 'Customer ID': 'customer_id', 'Country': 'country'}


## 3. Apply the mapping

Rename columns and apply the business rules to the standard format.

In [3]:
def apply_mapping(raw, mapping):
    """Convert one retailer's raw data into the standard transaction format.

    The rules come from the retailer's mapping file, so the same function
    works for any retailer that has a mapping file.
    """
    rules = mapping["rules"]
    df = raw.rename(columns=mapping["column_mapping"])[list(mapping["column_mapping"].values())].copy()
    df["channel"] = mapping["channel"]
    log = {"raw rows": len(df)}

    if rules["drop_exact_duplicates"]:
        df = df.drop_duplicates()
        log["after removing duplicates"] = len(df)

    df["order_id"] = df["order_id"].astype("string")
    df["product_id"] = df["product_id"].astype("string")

    # Cancellations are not purchases, but we keep them separately:
    # frequent cancelling may signal an unhappy customer.
    is_cancelled = df["order_id"].str.startswith(rules["cancelled_order_prefix"])
    cancellations = df[is_cancelled & df["customer_id"].notna()][
        ["customer_id", "order_id", "order_date", "product_id", "quantity", "unit_price"]].copy()
    cancellations["line_value"] = cancellations["quantity"] * cancellations["unit_price"]  # negative
    df = df[~is_cancelled]
    log["after removing cancellations"] = len(df)

    if rules["require_customer_id"]:
        df = df[df["customer_id"].notna()]
        log["after requiring a customer ID"] = len(df)

    non_product = df["product_id"].isin(rules["non_product_codes"])
    for prefix in rules["non_product_code_prefixes"]:
        non_product = non_product | df["product_id"].str.startswith(prefix)
    df = df[~non_product]
    log["after removing non-product codes"] = len(df)

    df = df[(df["quantity"] >= rules["min_quantity"]) & (df["unit_price"] >= rules["min_unit_price"])]
    log["after removing invalid quantity/price"] = len(df)

    df["customer_id"] = df["customer_id"].astype(int).astype(str)
    cancellations["customer_id"] = cancellations["customer_id"].astype(int).astype(str)
    df["line_value"] = df["quantity"] * df["unit_price"]
    return df.reset_index(drop=True), cancellations.reset_index(drop=True), log


transactions, cancellations, log = apply_mapping(raw, mapping)
pd.Series(log, name="rows")

raw rows                                 1067371
after removing duplicates                1033036
after removing cancellations             1013932
after requiring a customer ID             779495
after removing non-product codes          776657
after removing invalid quantity/price     776579
Name: rows, dtype: int64

## 4. Quality checks

Confirm required fields, sensible values and that revenue totals reconcile.

In [4]:
checks = {
    "every row has a customer ID": transactions["customer_id"].notna().all(),
    "every row has an order date": transactions["order_date"].notna().all(),
    "all quantities are positive": (transactions["quantity"] > 0).all(),
    "all prices are positive": (transactions["unit_price"] > 0).all(),
    "no cancelled orders remain": (~transactions["order_id"].str.startswith("C")).all(),
    "no duplicate rows remain": not transactions.duplicated().any(),
    "dates are within the expected range": transactions["order_date"].between("2009-12-01", "2011-12-31").all(),
}
for name, passed in checks.items():
    print("PASS" if passed else "FAIL", "-", name)
assert all(checks.values()), "A quality check failed - stop and investigate before modelling."

print()
print("Customers:", transactions["customer_id"].nunique())
print("Orders:", transactions["order_id"].nunique())
print("Products:", transactions["product_id"].nunique())
print("Date range:", transactions["order_date"].min(), "to", transactions["order_date"].max())
print("Revenue kept: £{:,.0f}".format(transactions["line_value"].sum()))
print("Cancelled orders kept as a signal:", cancellations["order_id"].nunique())

PASS - every row has a customer ID
PASS - every row has an order date
PASS - all quantities are positive
PASS - all prices are positive
PASS - no cancelled orders remain
PASS - no duplicate rows remain
PASS - dates are within the expected range

Customers: 5852
Orders: 36594
Products: 4620
Date range: 2009-12-01 07:45:00 to 2011-12-09 12:50:00
Revenue kept: £17,068,583
Cancelled orders kept as a signal: 7901


## 5. Choose the cut-off date

Split history (features) from the future window (churn label).

In [5]:
# Churn = no purchase in the next 180 days.
# To test the model honestly, we train on an earlier snapshot and test on a later one,
# so the model never sees the future it is asked to predict.
WINDOW_DAYS = 180
window = pd.Timedelta(days=WINDOW_DAYS)

data_end = transactions["order_date"].max().normalize() + pd.Timedelta(days=1)
TEST_CUTOFF = data_end - window           # label window runs to the end of the data
TRAIN_CUTOFF = TEST_CUTOFF - window       # train label window ends where test begins
SCORE_CUTOFF = data_end                   # "today": customers we would act on now

print("Train snapshot: history before", TRAIN_CUTOFF.date(), "| label window to", (TRAIN_CUTOFF + window).date())
print("Test snapshot:  history before", TEST_CUTOFF.date(), "| label window to", (TEST_CUTOFF + window).date())
print("Score snapshot: history before", SCORE_CUTOFF.date(), "| no label (the future is unknown)")

Train snapshot: history before 2010-12-15 | label window to 2011-06-13
Test snapshot:  history before 2011-06-13 | label window to 2011-12-10
Score snapshot: history before 2011-12-10 | no label (the future is unknown)


## 6. Build customer features

Recency, frequency, monetary value and other behaviour, using data before the cut-off only.

In [6]:
NET_CANCELLATIONS = mapping["rules"].get("net_cancellations_against_spend", False)

def build_customer_snapshot(tx, cancels, cutoff, window, with_label=True):
    """One row per customer, using only data before the cut-off date."""
    hist = tx[tx["order_date"] < cutoff]
    orders = (hist.groupby(["customer_id", "order_id"])
                  .agg(order_date=("order_date", "min"), order_value=("line_value", "sum"))
                  .reset_index())
    g = orders.groupby("customer_id")

    f = pd.DataFrame({
        "recency_days": (cutoff - g["order_date"].max()).dt.days,
        "tenure_days": (cutoff - g["order_date"].min()).dt.days,
        "frequency": g["order_id"].nunique(),
        "monetary": g["order_value"].sum(),
        "avg_order_value": g["order_value"].mean(),
    })
    recent = orders[orders["order_date"] >= cutoff - pd.Timedelta(days=90)].groupby("customer_id")
    f["orders_last_90d"] = recent["order_id"].nunique()
    f["spend_last_90d"] = recent["order_value"].sum()
    f["unique_products"] = hist.groupby("customer_id")["product_id"].nunique()
    f["days_per_order"] = f["tenure_days"] / f["frequency"]
    past_cancels = cancels[cancels["order_date"] < cutoff].groupby("customer_id")
    f["cancelled_orders"] = past_cancels["order_id"].nunique()
    f["cancelled_value"] = past_cancels["line_value"].sum()      # negative amounts
    f = f.fillna({"orders_last_90d": 0, "spend_last_90d": 0, "cancelled_orders": 0, "cancelled_value": 0})

    # Some customers placed very large orders and cancelled them minutes later.
    # Netting cancellations against spend stops those orders inflating a customer's value.
    if NET_CANCELLATIONS:
        f["monetary"] = (f["monetary"] + f["cancelled_value"]).clip(lower=0)
        f["avg_order_value"] = f["monetary"] / f["frequency"]
    f["cancel_rate"] = f["cancelled_orders"] / (f["frequency"] + f["cancelled_orders"])
    main_country = hist.groupby("customer_id")["country"].agg(lambda s: s.mode().iloc[0])
    f["is_uk"] = (main_country == "United Kingdom").astype(int)

    if with_label:
        future = tx[(tx["order_date"] >= cutoff) & (tx["order_date"] < cutoff + window)]
        f["churned"] = (~f.index.isin(future["customer_id"].unique())).astype(int)
        future_spend = future.groupby("customer_id")["line_value"].sum().reindex(f.index).fillna(0)
        if NET_CANCELLATIONS:
            future_cancels = cancels[(cancels["order_date"] >= cutoff) & (cancels["order_date"] < cutoff + window)]
            future_spend += future_cancels.groupby("customer_id")["line_value"].sum().reindex(f.index).fillna(0)
        f["future_spend"] = future_spend.clip(lower=0)

    f["snapshot_cutoff"] = cutoff
    return f.reset_index()

## 7. Create the churn label

Did the customer buy again in the future window?

In [7]:
train = build_customer_snapshot(transactions, cancellations, TRAIN_CUTOFF, window)
test = build_customer_snapshot(transactions, cancellations, TEST_CUTOFF, window)
score = build_customer_snapshot(transactions, cancellations, SCORE_CUTOFF, window, with_label=False)

for name, snap in [("train", train), ("test", test)]:
    print(f"{name}: {len(snap):,} customers, churn rate {snap['churned'].mean():.1%}")
print(f"score: {len(score):,} customers (current customers to act on)")
test.head()

train: 4,295 customers, churn rate 54.1%
test: 4,952 customers, churn rate 48.1%
score: 5,852 customers (current customers to act on)


,customer_id,recency_days,tenure_days,frequency,monetary,avg_order_value,orders_last_90d,spend_last_90d,unique_products,days_per_order,cancelled_orders,cancelled_value,cancel_rate,is_uk,churned,future_spend,snapshot_cutoff
0,12346,145,467,3,0.00,0.000,0.0,0.00,25,155.666667,5.0,-77608.20,0.625000,1,1,0.00,2011-06-13
1,12347,3,224,5,2817.48,563.496,2.0,1018.77,101,44.800000,0.0,0.00,0.000000,0,0,2104.05,2011-06-13
2,12348,68,258,4,1388.40,347.100,1.0,327.00,24,64.500000,0.0,0.00,0.000000,0,0,270.00,2011-06-13
3,12349,227,409,2,2196.99,1098.495,0.0,0.00,89,204.500000,1.0,-24.15,0.333333,0,0,1457.55,2011-06-13
4,12350,130,130,1,294.40,294.400,0.0,0.00,16,130.000000,0.0,0.00,0.000000,0,1,0.00,2011-06-13


## 8. Save the customer table



In [8]:
train["snapshot"] = "train"
test["snapshot"] = "test"
score["snapshot"] = "score"
customers = pd.concat([train, test, score], ignore_index=True)

transactions.to_parquet("../data/processed/transactions_clean.parquet", index=False)
customers.to_parquet("../data/processed/customer_snapshots.parquet", index=False)
print("Saved", len(transactions), "clean transactions and", len(customers), "customer snapshot rows")

Saved 776579 clean transactions and 15099 customer snapshot rows
